<a href="https://colab.research.google.com/github/tommbk50-hub/Cheminformatics/blob/main/AiZynthFinder_with_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Install Dependencies & Download Models
import os

# Install all Python dependencies in a single pip invocation
!pip install -q rdkit onnxruntime tables scipy pydantic deprecated jinja2 networkx pillow requests tqdm pyyaml rdchiral paretoset marshmallow jedi Reaction-Utils google-genai markdown
!pip install -q --ignore-requires-python --no-deps aizynthfinder

# Download USPTO models and ZINC stock only if not already present
if not os.path.exists("data/config.yml"):
    !mkdir -p data && download_public_data data
else:
    print("✅ Models and ZINC stock already present in ./data")

In [ ]:
# @title 2. Core Definitions: SMARTS Library, Reaction Classifiers & Gemini Helpers
import os
import json
import logging
from pydantic import BaseModel
from rdkit import Chem
from google import genai
from google.genai import types

# --- 1. API Key & Client Singleton ---
def get_gemini_client():
    """Returns a cached genai.Client instance if KDD_GEMINI_API_KEY is available."""
    if "_GEMINI_CLIENT" in globals() and globals()["_GEMINI_CLIENT"] is not None:
        return globals()["_GEMINI_CLIENT"]

    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")

    if api_key:
        globals()["_GEMINI_CLIENT"] = genai.Client(api_key=api_key)
        return globals()["_GEMINI_CLIENT"]
    return None

# --- 2. Medicinal Chemistry SMARTS Dictionary ---
# Pipe '|' separates sub-patterns so RDKit compiles each valid SMARTS string into a list
RAW_SMARTS = {
    "boronic": "[#5](O)O|[#5]1O[#6][#6]O1",
    "stannane": "[#50]",
    "zinc_magnesium": "[#12,#30]",
    "silicon_coupling": "[#14](F)|[#14](O)",
    "terminal_alkyne": "[CX2]#[CX2H1]",
    "internal_alkyne": "[CX2]#[CX2;!H1]",
    "alkene": "[CX3]=[CX3]",
    "aryl_halide_pseudohalide": "c[Cl,Br,I]|cOS(=O)(=O)C(F)(F)F",
    "alkyl_lg": "[CX4][Cl,Br,I]|[#6]OS(=O)(=O)[#6]",
    "carboxylic_acid": "[CX3](=O)[OX2H1]",
    "ester": "[#6][CX3](=O)[OX2][#6]",
    "benzyl_ester": "[#6][CX3](=O)[OX2]Cc1ccccc1",
    "acyl_halide": "[CX3](=O)[Cl,Br,F]",
    "anhydride": "[CX3](=O)O[CX3](=O)",
    "amide": "[#6][CX3](=O)[NX3]",
    "primary_amide": "[CX3](=O)[NX3H2]",
    "thioamide": "[#6](=[#16])[#7]",
    "amine_1_2": "[NX3;H2,H1;!$(NC=O);!$(NS=O)]",
    "tert_amine": "[NX3;H0;!$(NC=O);!$(NS=O)]",
    "aryl_amine": "c[NX3]",
    "nitro": "[$([NX3](=O)=O),$([NX3+](=O)[O-])]",
    "nitrile": "[CX2]#[NX1]",
    "azide": "[NX2]=[NX2+]=[NX1-]|[NX1-]=[NX2+]=[NX2]",
    "triazole": "c1ncnn1|c1nnn[nH]1",
    "tetrazole": "c1nnnn1",
    "thiazole": "c1ncsc1",
    "imidazole_oxazole": "c1nc[nH,o]c1",
    "indole": "c1ccc2[nH]ccc2c1",
    "aldehyde_ketone": "[CX3H1](=O)[#6]|[CX3](=O)([#6])[#6]",
    "alcohol": "[CX4][OX2H1]",
    "phenol": "c[OX2H1]",
    "ether": "[#6][OX2][#6]",
    "aryl_ether": "c[OX2][CX4]",
    "epoxide": "C1OC1",
    "sulfonamide": "S(=O)(=O)[NX3]",
    "sulfonyl_chloride": "S(=O)(=O)[Cl]",
    "thiol": "[#6][SX2H1]",
    "thioether": "[#6][SX2][#6]",
    "sulfone_sulfoxide": "[#16X4](=O)(=O)|[#16X3](=O)",
    "boc": "[NX3]C(=O)OC(C)(C)C",
    "cbz": "[NX3]C(=O)OCc1ccccc1",
    "silyl_ether": "[#8][#14]([#6])([#6])[#6]",
    "phosphorus_ylide_phosphonate": "[#15]",
}

MEDCHEM_SMARTS = {
    k: [Chem.MolFromSmarts(p) for p in v.split("|") if Chem.MolFromSmarts(p) is not None]
    for k, v in RAW_SMARTS.items()
}

def has_fg(mols, key):
    """Checks if any molecule in `mols` matches any SMARTS pattern for `key`."""
    pats = MEDCHEM_SMARTS.get(key, [])
    return any(m is not None and m.HasSubstructMatch(pat) for m in mols for pat in pats)

# --- 3. Local RDKit Heuristic Classifier (with Conditions) ---
# Ordered list of (Condition_Lambda(R, P), Reaction_Name, Typical_Conditions)
RDKIT_REACTION_RULES = [
    # 1. Cross-Couplings
    (lambda R, P: has_fg(R, "boronic") and (has_fg(R, "aryl_halide_pseudohalide") or has_fg(R, "alkyl_lg")),
     "Suzuki-Miyaura Cross-Coupling", "Pd(dppf)Cl2 or Pd(PPh3)4, K2CO3, Dioxane/H2O, 80-100 °C"),
    (lambda R, P: has_fg(R, "boronic") and (has_fg(R, "amine_1_2") or has_fg(R, "phenol")),
     "Chan-Lam Coupling", "Cu(OAc)2, Pyridine or Et3N, DCM, O2 (air), RT"),
    (lambda R, P: has_fg(R, "stannane") and has_fg(R, "aryl_halide_pseudohalide"),
     "Stille Cross-Coupling", "Pd(PPh3)4, LiCl, DMF or Dioxane, 90 °C"),
    (lambda R, P: has_fg(R, "terminal_alkyne") and has_fg(R, "aryl_halide_pseudohalide") and has_fg(P, "internal_alkyne"),
     "Sonogashira Cross-Coupling", "Pd(PPh3)2Cl2, CuI, Et3N, THF/DMF, RT-60 °C"),
    (lambda R, P: has_fg(R, "aryl_halide_pseudohalide") and has_fg(R, "amine_1_2") and has_fg(P, "aryl_amine"),
     "Buchwald-Hartwig Amination / SNAr", "Pd2(dba)3, XPhos/BINAP, NaOtBu, Toluene, 100 °C"),
    (lambda R, P: has_fg(P, "boronic") and has_fg(R, "aryl_halide_pseudohalide") and not has_fg(R, "boronic"),
     "Miyaura Borylation", "B2pin2, Pd(dppf)Cl2, KOAc, Dioxane, 90 °C"),
    (lambda R, P: has_fg(R, "aryl_halide_pseudohalide") and not has_fg(P, "aryl_halide_pseudohalide") and has_fg(R, "alkene") and has_fg(P, "alkene"),
     "Mizoroki-Heck Reaction", "Pd(OAc)2, P(o-tol)3, Et3N, DMF/MeCN, 100 °C"),
    (lambda R, P: has_fg(R, "zinc_magnesium") and has_fg(R, "aryl_halide_pseudohalide"),
     "Kumada / Negishi Cross-Coupling", "Pd(PPh3)4 or Ni(dppe)Cl2, THF, RT-65 °C"),

    # 2. Deprotections
    (lambda R, P: has_fg(R, "boc") and not has_fg(P, "boc"),
     "N-Boc Deprotection (Acidolysis)", "TFA / DCM or 4M HCl in Dioxane, RT"),
    (lambda R, P: has_fg(R, "cbz") and not has_fg(P, "cbz"),
     "N-Cbz Deprotection (Hydrogenolysis)", "H2, Pd/C, MeOH or EtOAc, RT"),
    (lambda R, P: has_fg(R, "silyl_ether") and not has_fg(P, "silyl_ether"),
     "Silyl Ether Deprotection (Desilylation)", "TBAF, THF, 0 °C to RT"),
    (lambda R, P: has_fg(P, "carboxylic_acid") and has_fg(R, "benzyl_ester") and not has_fg(R, "carboxylic_acid"),
     "Benzyl Ester Cleavage (Hydrogenolysis / Hydrolysis)", "H2, Pd/C in MeOH/THF (or LiOH, THF/H2O)"),
    (lambda R, P: has_fg(P, "carboxylic_acid") and has_fg(R, "ester") and not has_fg(R, "carboxylic_acid"),
     "Ester Hydrolysis / Saponification", "LiOH or NaOH, THF / MeOH / H2O, RT-50 °C"),
    (lambda R, P: has_fg(P, "phenol") and has_fg(R, "aryl_ether") and not has_fg(R, "phenol"),
     "Aryl Ether Cleavage / O-Dealkylation", "BBr3, DCM, -78 °C to RT"),

    # 3. Acylations, Amide & Sulfonamide Couplings, Esterifications
    (lambda R, P: has_fg(P, "amide") and not has_fg(R, "amide") and has_fg(R, "amine_1_2") and (has_fg(R, "acyl_halide") or has_fg(R, "anhydride")),
     "Schotten-Baumann Acylation / Amide Coupling", "Et3N or DIPEA, DCM, 0 °C to RT"),
    (lambda R, P: has_fg(P, "amide") and not has_fg(R, "amide") and has_fg(R, "amine_1_2") and has_fg(R, "carboxylic_acid"),
     "Amide Bond Coupling", "HATU or EDC/HOBt, DIPEA, DMF, RT"),
    (lambda R, P: has_fg(P, "sulfonamide") and has_fg(R, "sulfonyl_chloride") and has_fg(R, "amine_1_2"),
     "Sulfonamide Formation", "Et3N or Pyridine, DCM, 0 °C to RT"),
    (lambda R, P: has_fg(P, "ester") and has_fg(R, "carboxylic_acid") and (has_fg(R, "alcohol") or has_fg(R, "phenol") or has_fg(R, "alkyl_lg")),
     "Esterification (Fischer / Steglich / O-Alkylation)", "EDC, DMAP, DCM (or K2CO3, DMF for alkyl halide)"),

    # 4. Heterocycles & Cycloadditions
    (lambda R, P: has_fg(P, "thiazole") and not has_fg(R, "thiazole") and has_fg(R, "thioamide"),
     "Hantzsch Thiazole Synthesis", "EtOH or DMF, reflux (70-90 °C)"),
    (lambda R, P: has_fg(P, "triazole") and not has_fg(R, "triazole") and has_fg(R, "azide"),
     "CuAAC 'Click' Cycloaddition", "CuSO4, Sodium Ascorbate, t-BuOH/H2O, RT"),
    (lambda R, P: has_fg(P, "tetrazole") and not has_fg(R, "tetrazole") and has_fg(R, "nitrile"),
     "[3+2] Azide-Nitrile Cycloaddition (Tetrazole Synthesis)", "NaN3, NH4Cl or Bu3SnN3, DMF/Toluene, 100-120 °C"),

    # 5. Redox, Substitutions & Functional Group Interconversions
    (lambda R, P: (has_fg(P, "amine_1_2") or has_fg(P, "tert_amine")) and has_fg(R, "aldehyde_ketone") and has_fg(R, "amine_1_2"),
     "Reductive Amination", "NaBH(OAc)3 or NaBH3CN, AcOH, DCE/MeOH, RT"),
    (lambda R, P: has_fg(P, "amine_1_2") and has_fg(R, "nitro") and not has_fg(P, "nitro"),
     "Nitro Reduction to Amine", "H2, Pd/C in MeOH or Fe, NH4Cl in EtOH/H2O, 80 °C"),
    (lambda R, P: has_fg(P, "aryl_ether") and has_fg(R, "phenol") and has_fg(R, "alkyl_lg"),
     "Williamson Ether Synthesis / O-Alkylation", "K2CO3 or Cs2CO3, DMF or MeCN, 60-80 °C"),
    (lambda R, P: has_fg(P, "aryl_ether") and has_fg(R, "phenol") and has_fg(R, "alcohol"),
     "Mitsunobu Etherification", "PPh3, DIAD/DEAD, THF, 0 °C to RT"),
    (lambda R, P: has_fg(P, "nitrile") and not has_fg(R, "nitrile") and has_fg(R, "primary_amide"),
     "Primary Amide Dehydration to Nitrile", "TFAA, Et3N, DCM, 0 °C to RT (or POCl3)"),
    (lambda R, P: has_fg(P, "nitrile") and not has_fg(R, "nitrile") and has_fg(R, "aryl_halide_pseudohalide"),
     "Pd-Catalyzed Aryl Cyanation", "Zn(CN)2, Pd(PPh3)4, DMF, 100 °C"),
]

# --- 4. Enriched Step Classification Schema & Local Chemistry Heuristics ---
REACTION_CLASSIFICATION_CACHE = {}

class StepClassification(BaseModel):
    forward_smiles: str
    reaction_name: str
    typical_conditions: str
    selectivity_warning: str = "None"          # #3: Competing reactive sites or side-reaction risks
    protecting_group_note: str = "None"        # #5: Suggested protection/deprotection choreography
    safety_green_score: int = 4                # #4: 1 (Hazardous/Toxic, e.g., stannanes, azides, DCM) to 5 (Green/Safe)
    greener_solvent_swap: str = "None"         # #4: Recommended green solvent alternative (e.g., EtOAc/2-MeTHF instead of DCM/DMF)
    is_chiral_transformation: bool = False     # #6: True if a stereocenter is formed or inverted
    stereochem_note: str = "Achiral / Racemic" # #6: Asymmetric catalyst suggestion if chiral

class BatchReactionResponse(BaseModel):
    reactions: list[StepClassification]

def _match_rdkit_rule(R, P):
    """Returns (reaction_name, typical_conditions) from RDKIT_REACTION_RULES."""
    for rule_fn, rxn_name, conditions in RDKIT_REACTION_RULES:
        if rule_fn(R, P):
            return rxn_name, conditions
    return "Template-Guided Disconnection", "Standard literature conditions"

def analyze_step_heuristics(forward_smi, rxn_name="", conditions=""):
    """Evaluates chemoselectivity (#3), safety/greenness (#4), protecting groups (#5), and stereochemistry (#6)."""
    defaults = {
        "selectivity_warning": "None",
        "protecting_group_note": "None",
        "safety_green_score": 4,
        "greener_solvent_swap": "None",
        "is_chiral_transformation": False,
        "stereochem_note": "Achiral / Racemic",
    }
    if not forward_smi or ">>" not in forward_smi:
        return defaults

    react_str, prod_str = [s.strip() for s in forward_smi.split(">>", 1)]
    R = [m for s in react_str.split(".") if s and (m := Chem.MolFromSmiles(s)) is not None]
    P = [m for s in prod_str.split(".") if s and (m := Chem.MolFromSmiles(s)) is not None]
    if not R and not P:
        return defaults

    if not rxn_name or not conditions:
        inferred_name, inferred_cond = _match_rdkit_rule(R, P)
        rxn_name = rxn_name or inferred_name
        conditions = conditions or inferred_cond

    rxn_lower = rxn_name.lower()
    cond_lower = conditions.lower()

    # 1. Chemoselectivity (#3) & Protecting Group Choreography (#5)
    warnings = []
    pg_notes = []

    electrophilic_step = (
        has_fg(R, "carboxylic_acid")
        or has_fg(R, "acyl_halide")
        or has_fg(R, "anhydride")
        or has_fg(R, "sulfonyl_chloride")
        or has_fg(R, "alkyl_lg")
        or has_fg(R, "epoxide")
        or any(k in rxn_lower for k in ("acyl", "amide", "sulfonamide", "esterif", "alkylation"))
    )
    if has_fg(R, "amine_1_2") and (has_fg(R, "alcohol") or has_fg(R, "phenol") or has_fg(R, "thiol")) and electrophilic_step:
        warnings.append(
            "Competing nucleophiles detected (unprotected N-H amine alongside -OH/-SH) — risk of O/S-acylation or over-alkylation."
        )
        pg_notes.append(
            "Consider N-Boc/N-Cbz or O-TBS silyl protection prior to coupling, followed by downstream deprotection."
        )

    is_organometallic_or_coupling = (
        has_fg(R, "boronic")
        or has_fg(R, "stannane")
        or has_fg(R, "zinc_magnesium")
        or any(k in rxn_lower for k in ("suzuki", "stille", "sonogashira", "kumada", "negishi", "heck", "borylation", "cyanation"))
    )
    if is_organometallic_or_coupling:
        if has_fg(R, "zinc_magnesium") and (
            has_fg(R, "aldehyde_ketone") or has_fg(R, "carboxylic_acid") or has_fg(R, "alcohol") or has_fg(R, "phenol") or has_fg(R, "amine_1_2")
        ):
            warnings.append(
                "Unprotected protic or carbonyl group (-OH, -NH, -COOH, C=O) will quench Grignard/organozinc reagent."
            )
            pg_notes.append(
                "Protect carbonyl as acetal, carboxylic acid as ester, and -OH/-NH as O-TBS/N-Boc prior to Kumada/Negishi coupling."
            )
        elif has_fg(R, "thiol"):
            warnings.append(
                "Unprotected thiol (-SH) risks transition-metal catalyst poisoning and competing C-S coupling."
            )
            pg_notes.append("Protect thiol (e.g., S-trityl) prior to Pd/Cu cross-coupling.")
        elif has_fg(R, "aldehyde_ketone") or has_fg(R, "carboxylic_acid") or (
            has_fg(R, "amine_1_2") and "buchwald" not in rxn_lower and "chan-lam" not in rxn_lower
        ):
            warnings.append(
                "Unprotected polar/electrophilic group (free acid, aldehyde/ketone, or N-H amine) may interfere with cross-coupling."
            )
            pg_notes.append(
                "Consider masking carboxylic acid as alkyl/benzyl ester, amine as N-Boc, or carbonyl as acetal prior to coupling."
            )

    halide_sites = sum(
        len(m.GetSubstructMatches(pat))
        for m in R
        for pat in MEDCHEM_SMARTS.get("aryl_halide_pseudohalide", [])
        if hasattr(m, "GetSubstructMatches")
    )
    if halide_sites > 1:
        warnings.append(
            "Multiple aryl halide/pseudohalide sites present — monitor stoichiometry for regioselectivity (Ar-I > Ar-OTf ≈ Ar-Br >> Ar-Cl)."
        )

    # Dedicated deprotection / protecting-group choreography notes (#5)
    if has_fg(R, "boc") and not has_fg(P, "boc"):
        pg_notes.append("N-Boc deprotection step; verify acid-labile groups tolerate TFA/HCl cleavage and pair with upstream Boc2O installation.")
    elif has_fg(R, "cbz") and not has_fg(P, "cbz"):
        pg_notes.append("N-Cbz hydrogenolysis step; ensure alkenes, alkynes, and aryl halides are compatible with H2/Pd-C.")
    elif has_fg(R, "silyl_ether") and not has_fg(P, "silyl_ether"):
        pg_notes.append("O-Silyl deprotection (TBAF); pairs with upstream TBSCl/TBDPSCl hydroxyl protection.")
    elif has_fg(P, "carboxylic_acid") and (has_fg(R, "benzyl_ester") or has_fg(R, "ester")) and not has_fg(R, "carboxylic_acid"):
        pg_notes.append("Late-stage ester cleavage unmasks carboxylic acid cleanly after upstream couplings.")
    elif not pg_notes and (has_fg(R, "boc") or has_fg(R, "cbz") or has_fg(R, "silyl_ether") or has_fg(R, "benzyl_ester")):
        pg_notes.append("Protecting group carried intact through this step; schedule downstream deprotection when unmasking is needed.")

    # 2. Green Chemistry & Lab Safety Scoring (#4)
    safety_score = 4
    swaps = []

    if has_fg(R, "stannane") or has_fg(P, "stannane") or "bu3sn" in cond_lower:
        safety_score = 1
        swaps.append("Replace toxic organotin (Stille) with boronic ester (Suzuki-Miyaura) in 2-MeTHF/H2O or EtOH/H2O")

    if has_fg(R, "azide") or has_fg(P, "azide") or "nan3" in cond_lower:
        safety_score = min(safety_score, 1 if ("nan3" in cond_lower or "100" in cond_lower) else 2)
        swaps.append("Explosion/azide hazard — use flow/blast shield and strictly avoid chlorinated solvents (never mix NaN3 with DCM)")

    if any(tok in cond_lower for tok in ("bbr3", "pocl3", "zn(cn)2", "diad", "dead")):
        safety_score = min(safety_score, 2)

    if has_fg(R, "acyl_halide") or has_fg(R, "sulfonyl_chloride") or "tfaa" in cond_lower or "tfa" in cond_lower:
        safety_score = min(safety_score, 3)

    if "dce" in cond_lower:
        safety_score = min(safety_score, 2)
        swaps.append("EtOAc, 2-MeTHF, or MeOH (replace carcinogenic 1,2-DCE)")
    if "dcm" in cond_lower or "ch2cl2" in cond_lower:
        safety_score = min(safety_score, 3)
        swaps.append("EtOAc, 2-MeTHF, or CPME (replace chlorinated DCM)")
    if "dmf" in cond_lower:
        safety_score = min(safety_score, 3)
        swaps.append("MeCN, EtOAc, DMSO, or Cyrene (replace reprotoxic DMF)")
    if "dioxane" in cond_lower:
        safety_score = min(safety_score, 3)
        swaps.append("2-MeTHF, CPME, or t-AmylOH/H2O (replace carcinogenic 1,4-dioxane)")

    if safety_score == 4 and not swaps and any(g in cond_lower for g in ("etoh", "h2o", "etoac", "meoh", "t-buoh")):
        safety_score = 5

    safety_score = max(1, min(5, int(safety_score)))

    # 3. 3D Stereochemistry / Chiral Center Awareness (#6)
    r_centers = [c for m in R for c in Chem.FindMolChiralCenters(m, includeUnassigned=True)]
    p_centers = [c for m in P for c in Chem.FindMolChiralCenters(m, includeUnassigned=True)]
    r_unassigned = sum(1 for _, lbl in r_centers if lbl == "?")
    p_unassigned = sum(1 for _, lbl in p_centers if lbl == "?")
    r_assigned = sorted(lbl for _, lbl in r_centers if lbl != "?")
    p_assigned = sorted(lbl for _, lbl in p_centers if lbl != "?")

    is_chiral_tx = False
    stereochem_note = "Achiral / Racemic"
    if (
        len(p_centers) > len(r_centers)
        or p_unassigned > r_unassigned
        or (len(p_centers) > 0 and ("mitsunobu" in rxn_lower or p_assigned != r_assigned))
    ):
        is_chiral_tx = True
        if has_fg(R, "aldehyde_ketone"):
            stereochem_note = (
                "New stereocenter from prochiral carbonyl/imine — use Noyori asymmetric hydrogenation, CBS reduction, or chiral transaminase instead of achiral NaBH4/NaBH(OAc)3."
            )
        elif "mitsunobu" in rxn_lower:
            stereochem_note = (
                "SN2 stereochemical inversion at chiral carbinol center — use enantiopure alcohol precursor and monitor enantiomeric excess (ee)."
            )
        elif p_unassigned > 0:
            stereochem_note = (
                "Unassigned stereocenter (?) formed — apply asymmetric catalyst/chiral ligand control or downstream chiral SFC/salt resolution."
            )
        else:
            stereochem_note = (
                "Stereocenter formed or modified — employ chiral catalyst/ligand control or chiral resolution to secure target enantiomer."
            )
    elif len(p_centers) > 0:
        stereochem_note = "Pre-existing stereocenter(s) retained — use mild base/temperature to avoid epimerization."

    return {
        "selectivity_warning": " ".join(warnings) if warnings else "None",
        "protecting_group_note": " ".join(pg_notes) if pg_notes else "None",
        "safety_green_score": safety_score,
        "greener_solvent_swap": "; ".join(dict.fromkeys(swaps)) if swaps else "None",
        "is_chiral_transformation": is_chiral_tx,
        "stereochem_note": stereochem_note,
    }

def _normalize_step_classification(raw_dict):
    """Validates and normalizes a step classification dictionary to all 9 StepClassification fields."""
    obj = StepClassification(**raw_dict)
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "dict"):
        return obj.dict()
    return dict(raw_dict)

def rdkit_classify_step(forward_smi):
    """Evaluates forward reaction SMILES (Reactants >> Product) against RDKit SMARTS rules and local critics."""
    if not forward_smi or ">>" not in forward_smi:
        return _normalize_step_classification({
            "forward_smiles": str(forward_smi or ""),
            "reaction_name": "Functional Group Transformation",
            "typical_conditions": "Standard literature conditions",
        })

    react_str, prod_str = [s.strip() for s in forward_smi.split(">>", 1)]
    R = [Chem.MolFromSmiles(s) for s in react_str.split(".") if s]
    P = [Chem.MolFromSmiles(s) for s in prod_str.split(".") if s]

    rxn_name, conditions = _match_rdkit_rule(R, P)
    heuristics = analyze_step_heuristics(forward_smi, rxn_name=rxn_name, conditions=conditions)
    return _normalize_step_classification({
        "forward_smiles": forward_smi,
        "reaction_name": rxn_name,
        "typical_conditions": conditions,
        **heuristics,
    })

def batch_classify_with_gemini(unique_forward_smiles, use_gemini=True):
    """Classifies uncached reaction steps in a batched Gemini call and updates REACTION_CLASSIFICATION_CACHE with 9-field dicts."""
    if not use_gemini:
        return REACTION_CLASSIFICATION_CACHE

    client = get_gemini_client()
    if client is None:
        return REACTION_CLASSIFICATION_CACHE

    uncached = [s for s in unique_forward_smiles if s not in REACTION_CLASSIFICATION_CACHE]
    if not uncached:
        return REACTION_CLASSIFICATION_CACHE

    prompt = (
        "You are an expert medicinal and process chemist. For each forward reaction SMILES "
        "(Reactants >> Product) below, return a structured StepClassification entry populating ALL 9 fields:\n"
        "1. forward_smiles: exact input SMILES string.\n"
        "2. reaction_name: specific Named Reaction or transformation class (e.g., 'Suzuki-Miyaura Cross-Coupling', 'Hantzsch Thiazole Synthesis').\n"
        "3. typical_conditions: concise, realistic reagents, catalysts, solvents, and temperature.\n"
        "4. selectivity_warning: competing reactive sites or side-reaction risks ('None' if clean).\n"
        "5. protecting_group_note: suggested protection/deprotection choreography ('None' if none needed).\n"
        "6. safety_green_score: integer 1 (hazardous/toxic, e.g., stannanes, azides, DCE) to 5 (green/benign, e.g., EtOH/H2O, EtOAc).\n"
        "7. greener_solvent_swap: greener solvent alternative such as EtOAc, 2-MeTHF, or CPME instead of DCM/DMF/Dioxane ('None' if already green).\n"
        "8. is_chiral_transformation: boolean True if a stereocenter is formed, unassigned, or inverted.\n"
        "9. stereochem_note: asymmetric catalyst/chiral resolution recommendation if chiral, else 'Achiral / Racemic'.\n\n"
        + "\n".join(f"- {smi}" for smi in uncached)
    )

    response = None
    last_err = None
    for model_name in ("gemini-3.8-flash", "gemini-2.5-flash"):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": BatchReactionResponse,
                    "temperature": 0.1,
                },
            )
            break
        except Exception as e:
            last_err = e

    if response is None:
        print(f"ℹ️ Gemini batch classification skipped ({last_err}); falling back to local RDKit SMARTS.")
        return REACTION_CLASSIFICATION_CACHE

    try:
        parsed = json.loads(response.text)
        for item in parsed.get("reactions", []):
            fwd_smi = item.get("forward_smiles", "")
            if not fwd_smi:
                continue
            local_fallback = rdkit_classify_step(fwd_smi)
            merged = dict(local_fallback)
            for k, v in item.items():
                if v is not None and v != "":
                    merged[k] = v
            merged["safety_green_score"] = max(1, min(5, int(merged.get("safety_green_score", 4))))
            REACTION_CLASSIFICATION_CACHE[fwd_smi] = _normalize_step_classification(merged)
    except Exception as e:
        print(f"ℹ️ Gemini batch classification parse skipped ({e}); falling back to local RDKit SMARTS.")

    return REACTION_CLASSIFICATION_CACHE

def to_forward_smiles(retro_smi):
    """Converts AiZynthFinder's Product>>Reactants string into Reactants >> Product."""
    if ">>" in retro_smi:
        prod_s, react_s = retro_smi.split(">>", 1)
        return f"{react_s.strip()} >> {prod_s.strip()}"
    return retro_smi

In [ ]:
# @title 3. Initialize AiZynthFinder Engine
import logging
from aizynthfinder.aizynthfinder import AiZynthFinder

# Silence verbose DEBUG/INFO output across the notebook
logging.getLogger("aizynthfinder").setLevel(logging.WARNING)
for handler in logging.getLogger("aizynthfinder").handlers:
    handler.setLevel(logging.WARNING)

if "finder" not in globals():
    finder = AiZynthFinder(configfile="data/config.yml")
    finder.stock.select("zinc")
    finder.expansion_policy.select("uspto")
    finder.filter_policy.select("uspto")
    finder.config.post_processing.min_routes = 10
    finder.config.post_processing.max_routes = 10
    print("✅ AiZynthFinder initialized with USPTO policies and ZINC stock.")
else:
    print("✅ AiZynthFinder instance already active in memory.")

In [ ]:
# @title 4. Interactive Target Molecule Sketcher (JSME) + Multi-Identifier Resolver
import re
import urllib.parse
import requests
import IPython
from google.colab import output
from rdkit import Chem
from rdkit.Chem import Draw

# 1. Chemical Identifier Resolver (SMILES, InChI, InChIKey, PubChem CID, DrugBank, ChEMBL, CAS, Name)
def _fetch_pubchem_smiles(namespace, identifier):
    """Queries PubChem PUG-REST for Canonical/Connectivity SMILES."""
    encoded = urllib.parse.quote(str(identifier).strip())
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{namespace}/{encoded}/property/CanonicalSMILES,ConnectivitySMILES/JSON"
    resp = requests.get(url, timeout=8)
    if resp.status_code == 200:
        props = resp.json().get("PropertyTable", {}).get("Properties", [])
        if props:
            p = props[0]
            return p.get("CanonicalSMILES") or p.get("ConnectivitySMILES") or p.get("SMILES")
    return None

def _resolve_chemical_identifier(query_str):
    """Resolves SMILES, InChI, InChIKey, PubChem CID, DrugBank ID, ChEMBL ID, CAS, or Drug Name."""
    q = (query_str or "").strip()
    if not q:
        return None, "Empty input"

    # A. Local RDKit Check: InChI string
    if q.startswith("InChI="):
        mol = Chem.MolFromInchi(q)
        if mol:
            return Chem.MolToSmiles(mol), "InChI (RDKit)"

    # B. Local RDKit Check: Direct SMILES (skip if it looks like a pure number/CID or word name)
    if not q.isdigit() and not re.match(r"^(DB\d+|CHEMBL\d+|[A-Z]{14}-[A-Z]{10}-[A-Z])$", q, re.I):
        mol = Chem.MolFromSmiles(q)
        if mol and mol.GetNumAtoms() > 0:
            return Chem.MolToSmiles(mol), "SMILES (RDKit)"

    # C. InChIKey (27 chars: 14-10-1)
    if re.match(r"^[A-Z]{14}-[A-Z]{10}-[A-Z]$", q, re.I):
        smi = _fetch_pubchem_smiles("inchikey", q.upper())
        if smi:
            return smi, f"InChIKey ({q.upper()})"

    # D. PubChem CID (handles '54732242', 'CID:54732242', 'PubChem CID 54732242', etc.)
    cid_match = re.match(r"^(?:pubchem[\s_-]*)?(?:cid[:=\s_-]*)?(\d+)$", q, re.I)
    if cid_match:
        cid = cid_match.group(1)
        smi = _fetch_pubchem_smiles("cid", cid)
        if smi:
            return smi, f"PubChem CID ({cid})"

    # E. ChEMBL ID (e.g., CHEMBL1234)
    if re.match(r"^CHEMBL\d+$", q, re.I):
        chembl_id = q.upper()
        try:
            r = requests.get(f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json", timeout=8)
            if r.status_code == 200:
                structs = r.json().get("molecule_structures") or {}
                smi = structs.get("canonical_smiles")
                if smi:
                    return smi, f"ChEMBL ({chembl_id})"
        except Exception:
            pass
        smi = _fetch_pubchem_smiles("name", chembl_id)
        if smi:
            return smi, f"ChEMBL via PubChem ({chembl_id})"

    # F. DrugBank ID (e.g., DB04868) -> PubChem Registry / Name / UniChem fallback
    if re.match(r"^DB\d{5}$", q, re.I):
        db_id = q.upper()
        smi = _fetch_pubchem_smiles("xref/RegistryID", db_id) or _fetch_pubchem_smiles("name", db_id)
        if smi:
            return smi, f"DrugBank ({db_id})"
        # Fallback via EBI UniChem API (DrugBank source ID = 2 -> ChEMBL source ID = 1)
        try:
            u_resp = requests.get(f"https://www.ebi.ac.uk/unichem/rest/src_compound_id/{db_id}/2/1", timeout=8)
            if u_resp.status_code == 200 and u_resp.json():
                chembl_hit = u_resp.json()[0].get("src_compound_id")
                if chembl_hit:
                    return _resolve_chemical_identifier(chembl_hit)
        except Exception:
            pass

    # G. Generic Drug Name, Synonym, or CAS Number -> PubChem Name Lookup + NCI CACTUS fallback
    smi = _fetch_pubchem_smiles("name", q)
    if smi:
        return smi, f"PubChem Name/Registry ('{q}')"

    try:
        cactus_url = f"https://cactus.nci.nih.gov/chemical/structure/{urllib.parse.quote(q)}/smiles"
        c_resp = requests.get(cactus_url, timeout=8)
        if c_resp.status_code == 200 and c_resp.text.strip():
            first_line = c_resp.text.strip().splitlines()[0].strip()
            if Chem.MolFromSmiles(first_line):
                return first_line, f"NCI CACTUS ('{q}')"
    except Exception:
        pass

    return None, f"Could not resolve identifier: '{q}'"

# 2. Colab Callbacks for Lookup & Target Saving
def _lookup_and_set_target(query_str):
    smi, source_label = _resolve_chemical_identifier(query_str)
    if not smi:
        return IPython.display.JSON({"error": source_label})

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return IPython.display.JSON({"error": f"Resolved invalid SMILES from {source_label}: {smi}"})

    canon_smiles = Chem.MolToSmiles(mol)
    if "finder" in globals():
        finder.target_smiles = canon_smiles

    return IPython.display.JSON({
        "smiles": canon_smiles,
        "source": source_label,
    })

def _set_target_smiles(smiles_str):
    smiles_str = (smiles_str or "").strip()
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None or not smiles_str:
        print(f"⚠️ Invalid or empty structure: '{smiles_str}'")
        return
    canon_smiles = Chem.MolToSmiles(mol)
    if "finder" in globals():
        finder.target_smiles = canon_smiles
    print(f"✅ Successfully set finder.target_smiles = '{canon_smiles}'")
    display(Draw.MolToImage(mol, size=(320, 200)))

output.register_callback("notebook.resolve_identifier", _lookup_and_set_target)
output.register_callback("notebook.set_target_smiles", _set_target_smiles)

# 3. Render Interactive Identifier Search + JSME Molecule Sketcher
html_code = """
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; width: 680px;">
  <div style="margin-bottom: 6px; font-weight: 600; font-size: 14px; color: #202124;">
    🔎 1. Search by Chemical Identifier (Drug Name, PubChem CID, DrugBank ID, ChEMBL ID, InChIKey, CAS, or SMILES):
  </div>
  <div style="display: flex; gap: 8px; margin-bottom: 6px;">
    <input id="chem-id-input" type="text"
           placeholder="e.g., Febuxostat | DB04868 | 134018 | CHEMBL175691 | BAXJSDGLLature... | CAS 144060-53-7"
           style="flex: 1; padding: 8px 10px; font-size: 13px; border: 1px solid #ccc; border-radius: 5px;" />
    <button id="lookup-id-btn"
            style="background-color: #1a73e8; color: white; border: none; padding: 8px 14px;
                   font-size: 13px; font-weight: 600; border-radius: 5px; cursor: pointer; white-space: nowrap;">
      🔍 Fetch Structure
    </button>
  </div>
  <div style="display: flex; gap: 6px; flex-wrap: wrap; align-items: center; margin-bottom: 10px; font-size: 12px; color: #5f6368;">
    <span>Quick examples:</span>
    <button onclick="quickLookup('Febuxostat')" style="border:1px solid #dadce0; background:#f8f9fa; border-radius:12px; padding:2px 9px; cursor:pointer; font-size:11px;">Name: Febuxostat</button>
    <button onclick="quickLookup('DB04868')" style="border:1px solid #dadce0; background:#f8f9fa; border-radius:12px; padding:2px 9px; cursor:pointer; font-size:11px;">DrugBank: DB04868 (Nilotinib)</button>
    <button onclick="quickLookup('5479530')" style="border:1px solid #dadce0; background:#f8f9fa; padding:2px 9px; border-radius:12px; cursor:pointer; font-size:11px;">PubChem CID: 5479530 (Naldemedine)</button>
    <button onclick="quickLookup('KTUFNOKKBVMGRW-UHFFFAOYSA-N')" style="border:1px solid #dadce0; background:#f8f9fa; padding:2px 9px; border-radius:12px; cursor:pointer; font-size:11px;">InChIKey: Imatinib</button>
  </div>

  <div style="margin-bottom: 6px; font-weight: 600; font-size: 14px; color: #202124;">
    ✏️ 2. Or Draw / Edit Structure Directly in the Sketcher:
  </div>
  <div id="jsme_container" style="width: 680px; height: 400px; border: 1px solid #ccc; border-radius: 6px;"></div>

  <div style="margin-top: 10px; display: flex; align-items: center; gap: 10px; width: 680px;">
    <input id="manual-smiles" type="text" placeholder="Active SMILES string..."
           style="flex: 1; padding: 8px; font-family: monospace; font-size: 13px; border: 1px solid #ccc; border-radius: 4px;"
           value="Cc1nc(C2=CC=C(OCC(C)C)C(C#N)=C2)sc1C(=O)O" />
    <button id="save-smiles-btn" style="background-color: #28a745; color: white; border: none;
            padding: 9px 16px; font-size: 13px; font-weight: bold; border-radius: 5px; cursor: pointer; white-space: nowrap;">
      ✅ Set as finder.target_smiles
    </button>
  </div>
  <div id="smiles-status" style="margin-top: 6px; font-family: monospace; font-size: 13px; color: #155724;"></div>
</div>

<script>
function initJSME() {
  if (typeof JSApplet !== "undefined") {
    window.jsmeApplet = new JSApplet.JSME("jsme_container", "680px", "400px");
    const defaultSmiles = document.getElementById("manual-smiles").value;
    window.jsmeApplet.readGenericMolecularInput(defaultSmiles);
    window.jsmeApplet.setCallBack("AfterStructureModified", function(event) {
      const s = window.jsmeApplet.smiles();
      if (s) document.getElementById("manual-smiles").value = s;
    });
  }
}

window.jsmeOnLoad = function() { initJSME(); };

if (!document.getElementById("jsme-script")) {
  const script = document.createElement("script");
  script.id = "jsme-script";
  script.src = "https://jsme-editor.github.io/dist/jsme/jsme.nocache.js";
  document.head.appendChild(script);
} else {
  setTimeout(initJSME, 300);
}

async function resolveIdentifier() {
  const query = document.getElementById("chem-id-input").value.trim();
  const statusEl = document.getElementById("smiles-status");
  const lookupBtn = document.getElementById("lookup-id-btn");
  if (!query) return;

  statusEl.style.color = "#1a73e8";
  statusEl.innerText = "⏳ Resolving '" + query + "' across RDKit, PubChem, ChEMBL, DrugBank & CACTUS...";
  lookupBtn.disabled = true;

  try {
    const res = await google.colab.kernel.invokeFunction("notebook.resolve_identifier", [query], {});
    const data = res.data["application/json"];
    if (data.error) {
      statusEl.style.color = "#d93025";
      statusEl.innerText = "⚠️ " + data.error;
    } else {
      document.getElementById("manual-smiles").value = data.smiles;
      if (window.jsmeApplet) {
        window.jsmeApplet.readGenericMolecularInput(data.smiles);
      }
      statusEl.style.color = "#155724";
      statusEl.innerText = "✅ Resolved via " + data.source + " → finder.target_smiles = '" + data.smiles + "'";
    }
  } catch (err) {
    statusEl.style.color = "#d93025";
    statusEl.innerText = "⚠️ Kernel communication error: " + err;
  } finally {
    lookupBtn.disabled = false;
  }
}

function quickLookup(val) {
  document.getElementById("chem-id-input").value = val;
  resolveIdentifier();
}

document.getElementById("lookup-id-btn").onclick = resolveIdentifier;
document.getElementById("chem-id-input").addEventListener("keydown", function(e) {
  if (e.key === "Enter") {
    e.preventDefault();
    resolveIdentifier();
  }
});

// Sync manual SMILES input box changes to JSME canvas on Enter or blur
document.getElementById("manual-smiles").addEventListener("change", function(e) {
  const smi = e.target.value.trim();
  if (smi && window.jsmeApplet) {
    window.jsmeApplet.readGenericMolecularInput(smi);
  }
});

document.getElementById("save-smiles-btn").onclick = function() {
  const statusEl = document.getElementById("smiles-status");
  const manualInput = document.getElementById("manual-smiles").value.trim();
  const jsmeSmiles = (window.jsmeApplet && window.jsmeApplet.smiles()) ? window.jsmeApplet.smiles().trim() : "";

  // Prefer the manual input if edited directly, otherwise read from JSME canvas
  const smiles = manualInput || jsmeSmiles;
  if (window.jsmeApplet && smiles) {
    window.jsmeApplet.readGenericMolecularInput(smiles);
  }
  statusEl.style.color = "#155724";
  statusEl.innerText = "Setting finder.target_smiles = " + smiles + " ...";
  google.colab.kernel.invokeFunction("notebook.set_target_smiles", [smiles], {});
};
</script>
"""
IPython.display.display(IPython.display.HTML(html_code))

In [ ]:
# @title 5. Run Retrosynthesis Search & Display Top 10 Classified Routes
from IPython.display import display, HTML

USE_GEMINI_CLASSIFIER = True  # Set False to force local RDKit SMARTS classification

if not getattr(finder, "target_smiles", None):
    raise ValueError("⚠️ Please set a target molecule in Cell 4 before running the search.")

# 1. Run MCTS tree search and extract up to 10 routes (once!)
finder.tree_search()
finder.build_routes()
stats = finder.extract_statistics()

sorted_routes = sorted(
    finder.routes,
    key=lambda r: r["score"].get("state score", 0.0),
    reverse=True
)[:10]

# 2. Collect unique reaction steps and batch-classify
unique_steps = list(dict.fromkeys(
    to_forward_smiles(rxn_node.reaction_smiles())
    for route in sorted_routes
    for rxn_node in route["reaction_tree"].reactions()
    if ">>" in rxn_node.reaction_smiles()
))

gemini_lookup = batch_classify_with_gemini(unique_steps, use_gemini=USE_GEMINI_CLASSIFIER)

# 3. Render Summary & Route Cards
classifier_mode = "Gemini 2.5 Flash + RDKit Fallback" if gemini_lookup else "Local RDKit MedChem SMARTS"
print(f"Target SMILES: {finder.target_smiles} | Classifier: {classifier_mode}")
print(f"Search Solved: {stats['is_solved']} | Displaying Top {len(sorted_routes)} of {len(finder.routes)} Routes\n" + "=" * 90)

for idx, route in enumerate(sorted_routes, start=1):
    score_val = route["score"].get("state score", 0.0)
    rxn_tree = route["reaction_tree"]

    display(HTML(f"<h3 style='margin-bottom:4px; color:#1a73e8;'>Route #{idx} &mdash; State Score: {score_val:.4f} (Solved: {rxn_tree.is_solved})</h3>"))
    display(rxn_tree.to_image())

    reactions_forward = list(reversed(list(rxn_tree.reactions())))
    print(f"Chemical Reaction Steps ({len(reactions_forward)} step{'s' if len(reactions_forward) != 1 else ''}, Left-to-Right Forward Order):")

    for step_idx, rxn_node in enumerate(reactions_forward, start=1):
        meta = getattr(rxn_node, "metadata", {}) or {}
        template_code = meta.get("template_code", meta.get("template_hash", "N/A"))
        policy_prob = meta.get("policy_probability", None)
        prob_str = f" | Policy Prob: {policy_prob:.3f}" if isinstance(policy_prob, (int, float)) else ""

        forward_smi = to_forward_smiles(rxn_node.reaction_smiles())
        rxn_name, conditions = gemini_lookup.get(forward_smi) or rdkit_classify_step(forward_smi)

        print(f"  Step {step_idx}: {rxn_name}  [Template ID: {template_code}{prob_str}]")
        print(f"          Suggested Conditions: {conditions}")
        print(f"          Forward SMILES:       {forward_smi}")
    print("-" * 90)

In [ ]:
# @title Chat with Gemini about the retrosynthesis routes
import os
import html
import markdown
import IPython
from google.colab import output, userdata
from google import genai
from google.genai import types

# 1. Build a context summary of the current Target & Top 10 AiZynthFinder Routes
def _build_retrosynthesis_context():
    if "finder" not in globals() or not getattr(finder, "target_smiles", None):
        return "No target molecule or routes have been generated in AiZynthFinder yet."

    lines = [f"Target Molecule SMILES: {finder.target_smiles}\n"]
    sorted_r = sorted(
        finder.routes,
        key=lambda r: r["score"].get("state score", 0.0),
        reverse=True
    )[:10]

    for idx, route in enumerate(sorted_r, start=1):
        score_val = route["score"].get("state score", 0.0)
        rxn_tree = route["reaction_tree"]
        lines.append(f"### Route #{idx} (State Score: {score_val:.4f}, Solved: {rxn_tree.is_solved})")
        reactions_forward = list(reversed(list(rxn_tree.reactions())))
        for s_idx, rxn_node in enumerate(reactions_forward, start=1):
            meta = getattr(rxn_node, "metadata", {}) or {}
            t_id = meta.get("template_code", meta.get("template_hash", "N/A"))
            retro_smi = rxn_node.reaction_smiles()
            prod_s, react_s = retro_smi.split(">>") if ">>" in retro_smi else ("", retro_smi)
            fwd_smi = f"{react_s} >> {prod_s}"
            name_hint = gemini_lookup.get(fwd_smi, ("Reaction Step", ""))[0] if "gemini_lookup" in globals() else "Step"
            lines.append(f"  - Step {s_idx} ({name_hint}, Template {t_id}): {fwd_smi}")
    return "\n".join(lines)

# 2. Persistent Multi-Turn Chat State
_CHAT_HISTORY = []  # Stores types.Content objects for multi-turn conversation

def _get_system_instruction():
    return f"""You are an expert medicinal and process chemist assistant integrated into an AiZynthFinder retrosynthesis notebook.
Your job is to help the user find accurate, peer-reviewed and patent-backed reaction conditions (reagents, catalysts, ligands, solvents, temperatures, reaction times, and workup/selectivity notes) for the retrosynthetic routes generated below.

CRITICAL INSTRUCTIONS:
1. Always use Google Search grounding to verify literature precedents, journal references (e.g., J. Med. Chem., Org. Process Res. Dev., J. Org. Chem., Tetrahedron Lett.), and patent procedures (USPTO / WO patents) for the exact target molecule or closely related scaffolds.
2. Provide specific stoichiometric equivalents, catalysts, solvents, and temperatures where reported.
3. Explicitly cite your references inline and list full citations at the end of your response.
4. Flag any chemoselectivity, regioselectivity, or functional group compatibility risks.
5. Output chemical formulas as plain text (e.g., Pd(PPh3)4, Na2CO3). DO NOT use LaTeX or MathJax anywhere in your response.

CURRENT AIZYNTHFINDER ROUTES IN NOTEBOOK:
{_build_retrosynthesis_context()}
"""

def _send_chat_message(user_text, selected_model, use_search):
    global _CHAT_HISTORY
    user_text = (user_text or "").strip()
    if not user_text:
        return IPython.display.JSON({"error": "Empty message"})

    api_key = None
    try:
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")

    if not api_key:
        return IPython.display.JSON({
            "error": "Missing GEMINI_API_KEY. Please add GEMINI_API_KEY to Google Colab Secrets (🔑 icon on the left sidebar)."
        })

    client = genai.Client(api_key=api_key)
    _CHAT_HISTORY.append(types.Content(role="user", parts=[types.Part.from_text(text=user_text)]))

    # Configure Google Search Grounding for live literature & patent lookup
    tools = [types.Tool(google_search=types.GoogleSearch())] if use_search else []
    config = types.GenerateContentConfig(
        system_instruction=_get_system_instruction(),
        tools=tools,
        temperature=0.2,
    )

    # Attempt selected model first (e.g. flash-3.8); gracefully fall back if model alias isn't live on endpoint
    candidate_models = [selected_model, "gemini-3-flash-preview", "gemini-2.5-flash"]
    used_model = selected_model
    response = None
    last_err = None

    for m in dict.fromkeys(candidate_models):
        try:
            response = client.models.generate_content(
                model=m,
                contents=_CHAT_HISTORY,
                config=config,
            )
            used_model = m
            break
        except Exception as e:
            last_err = str(e)

    if response is None:
        _CHAT_HISTORY.pop()  # Remove failed turn
        return IPython.display.JSON({"error": f"API Error: {last_err}"})

    reply_text = response.text or "(No text returned)"
    _CHAT_HISTORY.append(types.Content(role="model", parts=[types.Part.from_text(text=reply_text)]))

    # Extract grounded web/literature citations if returned by Google Search tool
    sources_html = ""
    try:
        cand = response.candidates[0]
        gm = getattr(cand, "grounding_metadata", None)
        chunks = getattr(gm, "grounding_chunks", None) if gm else None
        if chunks:
            links = []
            for c in chunks:
                web = getattr(c, "web", None)
                if web and getattr(web, "uri", None):
                    title = html.escape(getattr(web, "title", None) or web.uri)
                    links.append(f"<li><a href='{web.uri}' target='_blank' style='color:#1a73e8;'>{title}</a></li>")
            if links:
                sources_html = (
                    "<div style='margin-top:10px; padding-top:8px; border-top:1px solid #e0e0e0; font-size:12px;'>"
                    "<b>📚 Grounded Literature & Web Sources:</b><ul style='margin:4px 0 0 18px; padding:0;'>"
                    + "".join(dict.fromkeys(links))
                    + "</ul></div>"
                )
    except Exception:
        pass

    note = ""
    if used_model != selected_model:
        note = f"<div style='font-size:11px; color:#b06000; margin-bottom:4px;'>ℹ️ Note: '{selected_model}' routed to active endpoint '{used_model}'.</div>"

    rendered_md = markdown.markdown(reply_text, extensions=["tables", "fenced_code"])
    return IPython.display.JSON({
        "html": note + rendered_md + sources_html,
        "model": used_model
    })

def _reset_chat_history():
    global _CHAT_HISTORY
    _CHAT_HISTORY = []
    return IPython.display.JSON({"status": "cleared"})

output.register_callback("notebook.gemini_retro_chat", _send_chat_message)
output.register_callback("notebook.gemini_retro_reset", _reset_chat_history)

# 3. Render Interactive Chat UI inside the Colab Cell Output
chat_ui_html = """
<style>
  .retro-chat-box {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    border: 1px solid #dadce0;
    border-radius: 10px;
    background: #ffffff;
    max-width: 960px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.06);
    overflow: hidden;
  }
  .retro-chat-header {
    background: #f8f9fa;
    padding: 12px 16px;
    border-bottom: 1px solid #dadce0;
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    justify-content: space-between;
    gap: 10px;
  }
  .retro-chat-messages {
    height: 440px;
    overflow-y: auto;
    padding: 16px;
    background: #fcfcfc;
    display: flex;
    flex-direction: column;
    gap: 12px;
  }
  .msg-user {
    align-self: flex-end;
    background: #e8f0fe;
    color: #174ea6;
    padding: 10px 14px;
    border-radius: 14px 14px 2px 14px;
    max-width: 78%;
    font-size: 14px;
    line-height: 1.45;
  }
  .msg-model {
    align-self: flex-start;
    background: #ffffff;
    color: #202124;
    border: 1px solid #e0e0e0;
    padding: 12px 16px;
    border-radius: 14px 14px 14px 2px;
    max-width: 88%;
    font-size: 14px;
    line-height: 1.55;
  }
  .msg-model table {
    border-collapse: collapse;
    margin: 8px 0;
    font-size: 13px;
  }
  .msg-model th, .msg-model td {
    border: 1px solid #ccc;
    padding: 6px 10px;
  }
  .retro-chat-input-area {
    padding: 12px 16px;
    border-top: 1px solid #dadce0;
    background: #f8f9fa;
    display: flex;
    gap: 10px;
    align-items: flex-end;
  }
  .retro-quick-btns {
    padding: 8px 16px;
    background: #f1f3f4;
    border-top: 1px solid #e8eaed;
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
    font-size: 12px;
  }
  .quick-chip {
    background: #fff;
    border: 1px solid #dadce0;
    border-radius: 16px;
    padding: 4px 10px;
    cursor: pointer;
    color: #3c4043;
  }
  .quick-chip:hover { background: #e8f0fe; border-color: #1a73e8; color: #1a73e8; }
</style>

<div class="retro-chat-box">
  <div class="retro-chat-header">
    <div>
      <strong style="font-size:15px; color:#1a73e8;">🔬 Gemini Retrosynthesis Literature & Conditions Assistant</strong>
      <div style="font-size:12px; color:#5f6368;">Context loaded with your Target SMILES & Top 10 AiZynthFinder Routes</div>
    </div>
    <div style="display:flex; align-items:center; gap:10px;">
      <label style="font-size:12px; font-weight:600; color:#3c4043;">Model:</label>
      <select id="gemini-model-select" style="padding:5px 8px; border-radius:5px; border:1px solid #ccc; font-size:13px;">
        <option value="flash-3.8" selected>flash-3.8 (Default)</option>
        <option value="gemini-3-flash-preview">gemini-3-flash-preview</option>
        <option value="gemini-3.1-pro-preview">gemini-3.1-pro-preview</option>
        <option value="gemini-2.5-flash">gemini-2.5-flash</option>
        <option value="gemini-2.5-pro">gemini-2.5-pro</option>
      </select>
      <label style="font-size:12px; display:flex; align-items:center; gap:4px; cursor:pointer; color:#3c4043;">
        <input type="checkbox" id="gemini-search-toggle" checked /> 📚 Literature Search
      </label>
      <button id="gemini-clear-btn" style="padding:5px 10px; border:1px solid #ccc; background:#fff; border-radius:5px; cursor:pointer; font-size:12px;">
        🗑️ Clear Chat
      </button>
    </div>
  </div>

  <div id="retro-chat-messages" class="retro-chat-messages">
    <div class="msg-model">
      👋 <b>Ready!</b> I have all <b>Top 10 AiZynthFinder routes</b> for your target molecule loaded in memory.
      Ask me to search the literature for exact reaction conditions, catalysts, yields, or patent procedures for any route or step below.
    </div>
  </div>

  <div class="retro-quick-btns">
    <span style="color:#5f6368; font-weight:600; align-self:center;">Quick prompts:</span>
    <button class="quick-chip" onclick="fillPrompt('Perform a literature search for the exact reaction conditions, reagents, yields, and references for each step of Route #1.')">Route #1 Literature Conditions</button>
    <button class="quick-chip" onclick="fillPrompt('Perform a literature search for the exact reaction conditions and references for Route #6 (Suzuki coupling, SNAr, and ester hydrolysis).')">Route #6 Conditions & Refs</button>
    <button class="quick-chip" onclick="fillPrompt('Compare Route #1 (Suzuki coupling) vs Route #2 (Hantzsch thiazole cyclization) in the published literature and patents for this compound. Which has higher yield and better scalability?')">Compare Route #1 vs Route #2</button>
  </div>

  <div class="retro-chat-input-area">
    <textarea id="retro-chat-input" rows="2" placeholder="Ask about reaction conditions, catalysts, literature precedents, or side-reactions (Press Enter to send, Shift+Enter for newline)..."
              style="flex:1; padding:10px; border:1px solid #ccc; border-radius:6px; font-family:inherit; font-size:14px; resize:vertical;"></textarea>
    <button id="retro-send-btn" style="background:#1a73e8; color:white; border:none; padding:10px 18px; border-radius:6px; font-weight:bold; cursor:pointer; height:42px;">
      Send ➤
    </button>
  </div>
</div>

<script>
function fillPrompt(text) {
  const input = document.getElementById("retro-chat-input");
  input.value = text;
  input.focus();
}

async function sendRetroMessage() {
  const inputEl = document.getElementById("retro-chat-input");
  const msgContainer = document.getElementById("retro-chat-messages");
  const modelSelect = document.getElementById("gemini-model-select").value;
  const useSearch = document.getElementById("gemini-search-toggle").checked;
  const sendBtn = document.getElementById("retro-send-btn");

  const text = inputEl.value.trim();
  if (!text) return;

  // Append user bubble
  const userDiv = document.createElement("div");
  userDiv.className = "msg-user";
  userDiv.innerText = text;
  msgContainer.appendChild(userDiv);
  inputEl.value = "";

  // Append loading bubble
  const loadingDiv = document.createElement("div");
  loadingDiv.className = "msg-model";
  loadingDiv.innerHTML = "<i>🔎 Searching chemical literature & generating response with <b>" + modelSelect + "</b>...</i>";
  msgContainer.appendChild(loadingDiv);
  msgContainer.scrollTop = msgContainer.scrollHeight;
  sendBtn.disabled = true;

  try {
    const result = await google.colab.kernel.invokeFunction(
      "notebook.gemini_retro_chat",
      [text, modelSelect, useSearch],
      {}
    );
    const data = result.data["application/json"];
    if (data.error) {
      loadingDiv.innerHTML = "<span style='color:#d93025;'>⚠️ " + data.error + "</span>";
    } else {
      loadingDiv.innerHTML = data.html;
    }
  } catch (err) {
    loadingDiv.innerHTML = "<span style='color:#d93025;'>⚠️ Error communicating with Colab kernel: " + err + "</span>";
  } finally {
    sendBtn.disabled = false;
    msgContainer.scrollTop = msgContainer.scrollHeight;
  }
}

document.getElementById("retro-send-btn").onclick = sendRetroMessage;
document.getElementById("retro-chat-input").addEventListener("keydown", function(e) {
  if (e.key === "Enter" && !e.shiftKey) {
    e.preventDefault();
    sendRetroMessage();
  }
});

document.getElementById("gemini-clear-btn").onclick = async function() {
  await google.colab.kernel.invokeFunction("notebook.gemini_retro_reset", [], {});
  document.getElementById("retro-chat-messages").innerHTML =
    '<div class="msg-model">🧹 <b>Chat history cleared.</b> Ask a new question about any of your retrosynthetic routes!</div>';
};
</script>
"""
IPython.display.display(IPython.display.HTML(chat_ui_html))